# Bank XYZ — Analytics & Insight Engineering
**Input:** Output CSV dari `1_preprocessing.ipynb` di folder `data/`

**Output:** File analitik tambahan siap dipakai dashboard

---
### Struktur Notebook
1. Load processed data
2. IPA Matrix lanjutan (per cabang, per panel)
3. Emotion Index analysis
4. Customer Segmentation profiling
5. Competitor deep analysis
6. Waiting time analysis
7. Digitalisasi analysis
8. Correlation & driver analysis
9. Export semua output

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── Fix path (hardcode) ───────────────────────────────────────
DATA_DIR = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data'

print("DATA_DIR:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))

# ── Load processed data ───────────────────────────────────────
master  = pd.read_csv(os.path.join(DATA_DIR, 'processed_bankxyz.csv'))
branch  = pd.read_csv(os.path.join(DATA_DIR, 'agg_branch.csv'))
prov    = pd.read_csv(os.path.join(DATA_DIR, 'agg_provinsi.csv'))
ipa_all = pd.read_csv(os.path.join(DATA_DIR, 'ipa_matrix.csv'))
emo     = pd.read_csv(os.path.join(DATA_DIR, 'emotion_summary.csv'))
segmen  = pd.read_csv(os.path.join(DATA_DIR, 'agg_segmen.csv'))
brand   = pd.read_csv(os.path.join(DATA_DIR, 'brand_perception.csv'))
comp    = pd.read_csv(os.path.join(DATA_DIR, 'competitor_benchmark.csv'))

print(f'Master: {master.shape}')
print(f'Branch: {len(branch)} cabang')
print(f'IPA: {len(ipa_all)} atribut')
print(f'Emotion: {len(emo)} emosi')
print('Data loaded ✓')

DATA_DIR: C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data
Exists: True
Master: (1730, 49)
Branch: 128 cabang
IPA: 110 atribut
Emotion: 16 emosi
Data loaded ✓


## 1. IPA Lanjutan — per Panel & per Provinsi

In [2]:
# ── IPA per Panel (Teller vs CS) ─────────────────────────────
# Load raw data untuk IPA per panel
RAW_DIR = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data'
df_raw = pd.read_excel(os.path.join(RAW_DIR, 'Deka_project_dataset_BankXYZ.xlsx'), header=1)

def parse_scale(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s in ['',' ','999']: return np.nan
    try:
        if s[0].isdigit(): return float(s[0])
    except: pass
    return np.nan

# IPA cols
IPA_MAP = {
    'Kantor Cabang': (
        [c for c in df_raw.columns if str(c).startswith('T_KC2_') and str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2],
        [c for c in df_raw.columns if str(c).startswith('T_KC2_') and str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0 and int(str(c).split('_')[-1])<=105]
    ),
    'Teller': (
        [c for c in df_raw.columns if str(c).startswith('T_TL3_') and str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2 and int(str(c).split('_')[-1])<=57],
        [c for c in df_raw.columns if str(c).startswith('T_TL3_') and str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0 and int(str(c).split('_')[-1])<=57]
    ),
    'Customer Service': (
        [c for c in df_raw.columns if str(c).startswith('T_CS3_') and str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==2 and int(str(c).split('_')[-1])<=69],
        [c for c in df_raw.columns if str(c).startswith('T_CS3_') and str(c).split('_')[-1].isdigit() and int(str(c).split('_')[-1])%3==0 and int(str(c).split('_')[-1])<=69]
    ),
}

# Parse numeric
df_raw_labels = pd.read_excel(os.path.join(RAW_DIR, 'Deka_project_dataset_BankXYZ.xlsx'), header=None)
labels_map = dict(zip(df_raw_labels.iloc[1].tolist(), df_raw_labels.iloc[0].tolist()))

ipa_panel_recs = []
PANEL_MAP = {
    'Teller': 'Teller (KUOTA 50%)',
    'CS': 'CS (KUOTA 50%)'
}

for panel, panel_raw in PANEL_MAP.items():
    df_panel = df_raw[df_raw['PANEL'] == panel_raw]
    for kat, (imp_cols, sat_cols) in IPA_MAP.items():
        n = min(len(imp_cols), len(sat_cols))
        for i in range(n):
            ic, sc = imp_cols[i], sat_cols[i]
            imp_v = df_panel[ic].apply(parse_scale).dropna()
            sat_v = df_panel[sc].apply(parse_scale).dropna()
            if len(imp_v) > 5 and len(sat_v) > 5:
                attr = str(labels_map.get(ic, f'Atribut {i+1}'))[:60]
                ipa_panel_recs.append({
                    'panel': panel, 'kategori': kat,
                    'atribut': attr, 'atribut_idx': i+1,
                    'importance': round(float(imp_v.mean()), 3),
                    'performance': round(float(sat_v.mean()), 3),
                    'gap': round(float(sat_v.mean() - imp_v.mean()), 3),
                })

ipa_panel_df = pd.DataFrame(ipa_panel_recs)

# Kuadran per panel
for panel in ipa_panel_df['panel'].unique():
    mask = ipa_panel_df['panel'] == panel
    sub  = ipa_panel_df[mask]
    imp_med = sub['importance'].median()
    sat_med = sub['performance'].median()
    def quad(row):
        hi = row['importance'] >= imp_med
        gd = row['performance'] >= sat_med
        if hi and gd:     return 'Keep Up'
        if hi and not gd: return 'Quick Win'
        if not hi and gd: return 'Possible Overkill'
        return 'Low Priority'
    ipa_panel_df.loc[mask, 'kuadran'] = sub.apply(quad, axis=1)

ipa_panel_df.to_csv(f'{DATA_DIR}/ipa_per_panel.csv', index=False)
print(f'IPA per Panel: {len(ipa_panel_df)} records')
print(ipa_panel_df.groupby(['panel','kuadran']).size().unstack(fill_value=0))

IPA per Panel: 154 records
kuadran  Keep Up  Low Priority  Possible Overkill  Quick Win
panel                                                       
CS            33            31                  7          6
Teller        25            22                 14         16


## 2. Emotion Index — Profil per Segmen & Panel

In [3]:
# ── Emotion per Segmen ────────────────────────────────────────
EMOTION_COLS_NUM = [c for c in master.columns if c.startswith('T_I1A_') and c.endswith('_num')]
EMOTION_LABELS = {
    'T_I1A_2_num':'Bahagia','T_I1A_5_num':'Percaya','T_I1A_8_num':'Dihargai',
    'T_I1A_11_num':'Diperhatikan','T_I1A_14_num':'Aman','T_I1A_17_num':'Fokus',
    'T_I1A_20_num':'Dimanjakan','T_I1A_23_num':'Tertarik','T_I1A_26_num':'Semangat',
    'T_I1A_29_num':'Tidak Puas','T_I1A_32_num':'Frustasi','T_I1A_35_num':'Kecewa',
    'T_I1A_38_num':'Tertekan','T_I1A_41_num':'Tidak Bahagia',
    'T_I1A_44_num':'Diabaikan','T_I1A_47_num':'Tergesa-gesa'
}
EMO_POS = ['T_I1A_2_num','T_I1A_5_num','T_I1A_8_num','T_I1A_11_num',
           'T_I1A_14_num','T_I1A_17_num','T_I1A_20_num','T_I1A_23_num','T_I1A_26_num']
EMO_NEG = ['T_I1A_29_num','T_I1A_32_num','T_I1A_35_num','T_I1A_38_num',
           'T_I1A_41_num','T_I1A_44_num','T_I1A_47_num']

# Emotion per segmen nasabah
emo_segmen_recs = []
for seg in master['customer_segment'].unique():
    sub = master[master['customer_segment'] == seg]
    for col, label in EMOTION_LABELS.items():
        if col in master.columns:
            vals = sub[col].dropna()
            if len(vals) > 0:
                tipe = 'positif' if col in EMO_POS else 'negatif'
                emo_segmen_recs.append({
                    'segmen': seg, 'emosi': label, 'tipe': tipe,
                    'mean_score': round(float(vals.mean()), 3),
                    'pct_strong': round(float((vals >= 5).mean() * 100), 1),
                    'n': len(vals)
                })

emo_segmen_df = pd.DataFrame(emo_segmen_recs)
emo_segmen_df.to_csv(f'{DATA_DIR}/emotion_per_segmen.csv', index=False)
print(f'Emotion per segmen: {len(emo_segmen_df)} records')

# Emotion per panel
emo_panel_recs = []
for panel in master['panel'].unique():
    sub = master[master['panel'] == panel]
    for col, label in EMOTION_LABELS.items():
        if col in master.columns:
            vals = sub[col].dropna()
            if len(vals) > 0:
                tipe = 'positif' if col in EMO_POS else 'negatif'
                emo_panel_recs.append({
                    'panel': panel, 'emosi': label, 'tipe': tipe,
                    'mean_score': round(float(vals.mean()), 3),
                    'pct_strong': round(float((vals >= 5).mean() * 100), 1),
                    'n': len(vals)
                })

emo_panel_df = pd.DataFrame(emo_panel_recs)
emo_panel_df.to_csv(f'{DATA_DIR}/emotion_per_panel.csv', index=False)
print(f'Emotion per panel: {len(emo_panel_df)} records')

Emotion per segmen: 80 records
Emotion per panel: 32 records


## 3. Customer Segmentation — Profil Lengkap

In [4]:
# ── Profil lengkap per segmen ─────────────────────────────────
def nps_score(series):
    s = series.dropna()
    if len(s) == 0: return np.nan
    return float(round(((s >= 9).sum() - (s <= 6).sum()) / len(s) * 100, 1))

seg_profile_recs = []
for seg in master['customer_segment'].unique():
    sub = master[master['customer_segment'] == seg]

    # Gender distribution
    gender_dist = sub['gender'].value_counts(normalize=True).mul(100).round(1).to_dict()

    # Usia distribution
    usia_dist = sub['usia_group'].value_counts(normalize=True).mul(100).round(1).to_dict()

    # Panel distribution
    panel_dist = sub['panel'].value_counts(normalize=True).mul(100).round(1).to_dict()

    # Emotion net
    emo_net = sub['emotion_net'].mean() if 'emotion_net' in sub.columns else np.nan

    seg_profile_recs.append({
        'segmen': seg,
        'n': len(sub),
        'pct': round(len(sub) / len(master) * 100, 1),
        'nps_score': nps_score(sub['nps_num']),
        'csi_mean': round(float(sub['csi_num'].mean()), 3),
        'loyalty_mean': round(float(sub['loyalty_num'].mean()), 3),
        'emotion_net': round(float(emo_net), 3) if not np.isnan(emo_net) else None,
        'pct_male': gender_dist.get('Laki-laki', gender_dist.get('Male', 0)),
        'pct_teller': panel_dist.get('Teller', 0),
        'top_usia': max(usia_dist, key=usia_dist.get) if usia_dist else '',
    })

seg_profile_df = pd.DataFrame(seg_profile_recs).sort_values('nps_score', ascending=False)
seg_profile_df.to_csv(f'{DATA_DIR}/segmen_profile.csv', index=False)
print('Segmen Profile:')
print(seg_profile_df[['segmen','n','pct','nps_score','csi_mean','emotion_net']].to_string())

Segmen Profile:
           segmen    n   pct  nps_score  csi_mean  emotion_net
0  Loyal Champion  316  18.3      100.0     5.965        4.626
1       Satisfied  434  25.1      100.0     5.965        4.286
2         Unknown  847  49.0       77.7     5.857        4.126
3         Passive  124   7.2        0.0     5.677        3.420
4         At Risk    9   0.5     -100.0     5.556        3.185


## 4. Waiting Time Analysis

In [5]:
# ── Waktu tunggu Teller & CS ──────────────────────────────────
# Cari kolom waiting time di raw data
wait_cols = [c for c in df_raw.columns if 'berapa lama' in str(labels_map.get(c,'')).lower()
             or 'menit' in str(labels_map.get(c,'')).lower()
             or 'waktu tunggu' in str(labels_map.get(c,'')).lower()]

print(f'Waiting time cols found: {len(wait_cols)}')
for c in wait_cols[:6]:
    print(f'  [{c}] {str(labels_map.get(c,""))[:70]}')
    print(f'  Sample: {df_raw[c].dropna().head(3).tolist()}')

# Waiting time cols Teller: T_TL1_, T_TL2_
tl_wait = [c for c in df_raw.columns if str(c).startswith('T_TL1_') or str(c).startswith('T_TL2_')]
cs_wait = [c for c in df_raw.columns if str(c).startswith('T_CS1_') or str(c).startswith('T_CS2_')]

print(f'\nTeller wait cols: {tl_wait[:5]}')
print(f'CS wait cols: {cs_wait[:5]}')

# Parse & analyze
wait_recs = []
for col, panel_name in [(tl_wait[0] if tl_wait else None, 'Teller'),
                         (cs_wait[0] if cs_wait else None, 'Customer Service')]:
    if col and col in df_raw.columns:
        vals = df_raw[col].dropna()
        label = str(labels_map.get(col,''))[:60]
        vc = vals.value_counts().head(10)
        for v, cnt in vc.items():
            wait_recs.append({
                'panel': panel_name, 'wait_category': str(v),
                'count': int(cnt), 'pct': round(cnt/len(vals)*100, 1)
            })

if wait_recs:
    wait_df = pd.DataFrame(wait_recs)
    wait_df.to_csv(f'{DATA_DIR}/waiting_time.csv', index=False)
    print('\nWaiting time analysis:')
    print(wait_df.head(10))
else:
    print('Waiting time data tidak ditemukan — skip')

Waiting time cols found: 5
  [S4] Sudah berapa lamakah Bapak/Ibu menjadi nasabah Bank XYZ?
  Sample: ['5 tahun atau lebih', '5 tahun atau lebih', '5 tahun atau lebih']
  [TL5] Berapa lama waktu tunggu/antri Teller Anda untuk kunjungan hari ini? _
  Sample: [0, 1, 5]
  [TL6] Berapa lamakah waktu tunggu/antri Teller yang Anda bisa terima/toleran
  Sample: [10, 1, 15]
  [CS5] Berapa lama waktu tunggu/antri CS Anda untuk kunjungan hari ini? ___ m
  Sample: [0, ' ', ' ']
  [CS6] Berapa lamakah waktu tunggu/antri CS yang Anda bisa terima/toleransi? 
  Sample: [10, ' ', ' ']

Teller wait cols: ['T_TL2_1', 'T_TL2_2', 'T_TL2_3', 'T_TL2_4', 'T_TL2_5']
CS wait cols: ['T_CS2_1', 'T_CS2_2', 'T_CS2_3', 'T_CS2_4', 'T_CS2_5']

Waiting time analysis:
              panel      wait_category  count   pct
0            Teller  6  SANGAT PENTING    936  54.1
1            Teller                       661  38.2
2            Teller                  5    130   7.5
3            Teller                  4      3   

## 5. Driver Analysis — Korelasi Touchpoint vs NPS

In [6]:
# ── Pearson correlation: setiap touchpoint vs NPS ─────────────
# Overall satisfaction cols vs NPS
ovr_cols = [c for c in master.columns if c.startswith('ovr_')]

driver_recs = []
LABEL_MAP = {
    'ovr_operasional':  'Operasional','ovr_parkir':'Parkir',
    'ovr_banking_hall': 'Banking Hall','ovr_toilet':'Toilet',
    'ovr_sekuriti':     'Sekuriti','ovr_teller':'Teller','ovr_cs':'Customer Service'
}

for col in ovr_cols:
    if col in master.columns:
        valid = master[['nps_num', col]].dropna()
        if len(valid) > 30:
            corr = valid['nps_num'].corr(valid[col])
            driver_recs.append({
                'touchpoint': LABEL_MAP.get(col, col),
                'correlation': round(float(corr), 4),
                'n': len(valid),
                'abs_corr': abs(round(float(corr), 4))
            })

# Tambahkan emotion net
if 'emotion_net' in master.columns:
    valid = master[['nps_num','emotion_net']].dropna()
    corr = valid['nps_num'].corr(valid['emotion_net'])
    driver_recs.append({
        'touchpoint': 'Emotion Net Score',
        'correlation': round(float(corr), 4),
        'n': len(valid),
        'abs_corr': abs(round(float(corr), 4))
    })

driver_df = pd.DataFrame(driver_recs).sort_values('abs_corr', ascending=False)
driver_df.to_csv(f'{DATA_DIR}/driver_analysis.csv', index=False)
print('Driver Analysis (Korelasi vs NPS):')
print(driver_df[['touchpoint','correlation','n']].to_string())

Driver Analysis (Korelasi vs NPS):
          touchpoint  correlation     n
1             Parkir       0.4142  1730
0        Operasional       0.4067  1730
2       Banking Hall       0.3853  1730
5             Teller       0.3716  1069
6   Customer Service       0.3672  1046
4           Sekuriti       0.3664  1730
3             Toilet       0.3280  1730
7  Emotion Net Score       0.3065  1730


## 6. NPS per Kombinasi Demografi

In [7]:
# ── NPS per gender × panel ────────────────────────────────────
nps_gender_panel = master.groupby(['gender','panel'])['nps_num'].apply(nps_score).reset_index()
nps_gender_panel.columns = ['gender','panel','nps_score']
nps_gender_panel.to_csv(f'{DATA_DIR}/nps_gender_panel.csv', index=False)
print('NPS Gender × Panel:')
print(nps_gender_panel)

# ── NPS per usia × panel ──────────────────────────────────────
nps_usia_panel = master.groupby(['usia_group','panel'])['nps_num'].apply(nps_score).reset_index()
nps_usia_panel.columns = ['usia_group','panel','nps_score']
nps_usia_panel.to_csv(f'{DATA_DIR}/nps_usia_panel.csv', index=False)
print('\nNPS Usia × Panel:')
print(nps_usia_panel)

# ── NPS per provinsi × panel ──────────────────────────────────
nps_prov_panel = master.groupby(['provinsi','panel'])['nps_num'].apply(nps_score).reset_index()
nps_prov_panel.columns = ['provinsi','panel','nps_score']
nps_prov_panel.to_csv(f'{DATA_DIR}/nps_prov_panel.csv', index=False)
print('\nNPS Prov × Panel (top 5):')
print(nps_prov_panel.head(10))

NPS Gender × Panel:
   gender               panel  nps_score
0    Pria      CS (KUOTA 50%)       80.6
1    Pria  Teller (KUOTA 50%)       82.4
2  Wanita      CS (KUOTA 50%)       78.8
3  Wanita  Teller (KUOTA 50%)       81.5

NPS Usia × Panel:
              usia_group               panel  nps_score
0           17 -19 tahun      CS (KUOTA 50%)       88.9
1           17 -19 tahun  Teller (KUOTA 50%)       80.0
2          20 - 25 tahun      CS (KUOTA 50%)       82.8
3          20 - 25 tahun  Teller (KUOTA 50%)       88.1
4          26 - 30 tahun      CS (KUOTA 50%)       74.2
5          26 - 30 tahun  Teller (KUOTA 50%)       83.6
6          31 - 35 tahun      CS (KUOTA 50%)       87.4
7          31 - 35 tahun  Teller (KUOTA 50%)       83.0
8          36 - 40 tahun      CS (KUOTA 50%)       74.2
9          36 - 40 tahun  Teller (KUOTA 50%)       81.7
10         41 - 45 tahun      CS (KUOTA 50%)       84.9
11         41 - 45 tahun  Teller (KUOTA 50%)       84.8
12         46 - 50 tahun    

## 7. Competitor Deep Analysis

In [8]:
# ── NPS per provinsi — XYZ vs Kompetitor ─────────────────────
def parse_nps(val):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    if s.startswith('10'): return 10.0
    try:
        if s[0].isdigit(): return float(s[0])
    except: pass
    return np.nan

df_raw['nps_xyz']  = df_raw['G1A'].apply(parse_nps)
df_raw['nps_komp'] = df_raw['G1C'].apply(parse_nps)

# Per provinsi
comp_prov_recs = []
for prov_name in df_raw['PROV'].unique():
    sub = df_raw[df_raw['PROV'] == prov_name]
    xyz_nps  = nps_score(sub['nps_xyz'])
    komp_nps = nps_score(sub['nps_komp'])
    comp_prov_recs.append({
        'provinsi': prov_name,
        'xyz_nps': xyz_nps,
        'komp_nps': komp_nps,
        'selisih': round(xyz_nps - komp_nps, 1) if komp_nps is not None and not np.isnan(komp_nps) else None,
        'n_xyz': int(sub['nps_xyz'].notna().sum()),
        'n_komp': int(sub['nps_komp'].notna().sum()),
    })

comp_prov_df = pd.DataFrame(comp_prov_recs).sort_values('xyz_nps', ascending=False)
comp_prov_df.to_csv(f'{DATA_DIR}/comp_nps_per_provinsi.csv', index=False)
print('Competitor NPS per Provinsi:')
print(comp_prov_df[['provinsi','xyz_nps','komp_nps','selisih']].to_string())

Competitor NPS per Provinsi:
              provinsi  xyz_nps  komp_nps  selisih
12    Sulawesi Selatan    100.0      68.2     31.8
5                 Bali     96.3     100.0     -3.7
7   Kalimantan Selatan     92.6       0.0     92.6
6           Jawa Barat     85.9      27.8     58.1
8     Sumatera Selatan     85.2      46.2     39.0
2               Banten     82.9      10.2     72.7
11                Riau     77.8      11.1     66.7
10             Lampung     77.8      22.2     55.6
1          Jawa Tengah     72.7       2.9     69.8
4           Jawa Timur     66.7      58.3      8.4
0          DKI Jakarta     65.1      36.4     28.7
13    Kalimantan Timur     59.3      31.6     27.7
9       Kepulauan Riau     55.6      18.5     37.1
3       Sumatera Utara     50.0      26.7     23.3


## 8. Summary Output

In [9]:
# ── List semua file output ────────────────────────────────────
print('=== OUTPUT FILES dari analytics.ipynb ===')
new_files = [
    ('ipa_per_panel.csv',        'IPA matrix per panel Teller/CS'),
    ('emotion_per_segmen.csv',   'Emotion index per segmen nasabah'),
    ('emotion_per_panel.csv',    'Emotion index per panel'),
    ('segmen_profile.csv',       'Profil lengkap tiap segmen nasabah'),
    ('waiting_time.csv',         'Distribusi waktu tunggu Teller & CS'),
    ('driver_analysis.csv',      'Korelasi touchpoint vs NPS'),
    ('nps_gender_panel.csv',     'NPS per gender × panel'),
    ('nps_usia_panel.csv',       'NPS per usia × panel'),
    ('nps_prov_panel.csv',       'NPS per provinsi × panel'),
    ('comp_nps_per_provinsi.csv','NPS XYZ vs kompetitor per provinsi'),
]

for fname, desc in new_files:
    fpath = f'{DATA_DIR}/{fname}'
    if os.path.exists(fpath):
        rows = sum(1 for _ in open(fpath)) - 1
        size = os.path.getsize(fpath)
        print(f'  ✓ {fname:<40} {rows:>5} baris  ({desc})')
    else:
        print(f'  ✗ {fname} — tidak dibuat (data tidak tersedia)')

print('\nAnalytics selesai — semua file siap dipakai dashboard!')

=== OUTPUT FILES dari analytics.ipynb ===
  ✓ ipa_per_panel.csv                          154 baris  (IPA matrix per panel Teller/CS)
  ✓ emotion_per_segmen.csv                      80 baris  (Emotion index per segmen nasabah)
  ✓ emotion_per_panel.csv                       32 baris  (Emotion index per panel)
  ✓ segmen_profile.csv                           5 baris  (Profil lengkap tiap segmen nasabah)
  ✓ waiting_time.csv                             9 baris  (Distribusi waktu tunggu Teller & CS)
  ✓ driver_analysis.csv                          8 baris  (Korelasi touchpoint vs NPS)
  ✓ nps_gender_panel.csv                         4 baris  (NPS per gender × panel)
  ✓ nps_usia_panel.csv                          16 baris  (NPS per usia × panel)
  ✓ nps_prov_panel.csv                          28 baris  (NPS per provinsi × panel)
  ✓ comp_nps_per_provinsi.csv                   14 baris  (NPS XYZ vs kompetitor per provinsi)

Analytics selesai — semua file siap dipakai dashboard!


In [10]:
import pandas as pd
import os

DATA_DIR = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data'

files = os.listdir(DATA_DIR)
print("Files:", sorted(files))

# Cek kolom tiap file penting
for f in ['processed_bankxyz.csv', 'agg_branch.csv', 'ipa_matrix.csv', 
          'emotion_summary.csv', 'brand_perception.csv', 'nps_competitor.csv']:
    df = pd.read_csv(os.path.join(DATA_DIR, f))
    print(f"\n{f}: {df.shape}")
    print(df.columns.tolist())

Files: ['Deka_project_dataset_BankXYZ.xlsx', 'agg_branch.csv', 'agg_gender.csv', 'agg_panel.csv', 'agg_provinsi.csv', 'agg_segmen.csv', 'agg_usia.csv', 'ai_config.json', 'auto_narrative.txt', 'brand_perception.csv', 'comp_nps_per_provinsi.csv', 'competitor_benchmark.csv', 'digitalisasi.csv', 'driver_analysis.csv', 'emotion_per_panel.csv', 'emotion_per_segmen.csv', 'emotion_summary.csv', 'ipa_matrix.csv', 'ipa_per_panel.csv', 'nps_competitor.csv', 'nps_gender_panel.csv', 'nps_prov_panel.csv', 'nps_usia_panel.csv', 'overall_satisfaction.csv', 'processed_bankxyz.csv', 'segmen_profile.csv', 'waiting_time.csv']

processed_bankxyz.csv: (1730, 49)
['SERIAL', 'provinsi', 'kota', 'cabang', 'panel', 'gender', 'usia', 'usia_group', 'lama_nasabah', 'frekuensi_transaksi', 'status_nikah', 'jumlah_anak', 'pendidikan', 'pekerjaan', 'pengeluaran', 'penghasilan', 'nps_num', 'csi_num', 'loyalty_num', 'nps_segment', 'emotion_positive_score', 'emotion_negative_score', 'emotion_net', 'customer_segment', 'la

In [11]:
# ── Switching Analysis ────────────────────────────────────────
df_raw = pd.read_excel(os.path.join(RAW_DIR, 'Deka_project_dataset_BankXYZ.xlsx'), header=1)
total = len(df_raw)

sw_recs = []

# Bank Simpan Utama (A1B)
for bank, count in df_raw['A1B'].value_counts().items():
    if str(bank).strip() and bank != '(Menolak)':
        sw_recs.append({
            'tipe': 'Bank Simpan Utama',
            'bank': str(bank).strip(),
            'n': int(count),
            'pct': round(count / total * 100, 1)
        })

# Bank Transaksi Utama (A1C)
for bank, count in df_raw['A1C'].value_counts().items():
    if str(bank).strip():
        sw_recs.append({
            'tipe': 'Bank Transaksi Utama',
            'bank': str(bank).strip(),
            'n': int(count),
            'pct': round(count / total * 100, 1)
        })

sw_df = pd.DataFrame(sw_recs)
sw_df.to_csv(os.path.join(DATA_DIR, 'switching_analysis.csv'), index=False)
print('switching_analysis.csv saved:')
print(sw_df)

switching_analysis.csv saved:
                    tipe                         bank     n   pct
0      Bank Simpan Utama                     Bank XYZ  1578  91.2
1      Bank Simpan Utama      Bank Central Asia (BCA)    49   2.8
2      Bank Simpan Utama  Bank Rakyat Indonesia (BRI)    32   1.8
3      Bank Simpan Utama                 Bank Mandiri    31   1.8
4      Bank Simpan Utama  Bank Negara Indonesia (BNI)    17   1.0
5      Bank Simpan Utama                      Lainnya    16   0.9
6      Bank Simpan Utama                 Bank Permata     2   0.1
7      Bank Simpan Utama              Bank CIMB Niaga     2   0.1
8      Bank Simpan Utama                   Bank Panin     1   0.1
9      Bank Simpan Utama                    Bank Mega     1   0.1
10  Bank Transaksi Utama                     Bank XYZ  1532  88.6
11  Bank Transaksi Utama      Bank Central Asia (BCA)    84   4.9
12  Bank Transaksi Utama                 Bank Mandiri    39   2.3
13  Bank Transaksi Utama  Bank Rakyat Indonesi

In [12]:
# ── Multibank Analysis ────────────────────────────────────────
df_raw['has_other_simpan']    = df_raw['A1XX'].notna() & (df_raw['A1XX'].astype(str).str.strip() != '')
df_raw['has_other_transaksi'] = df_raw['A1X'].notna()  & (df_raw['A1X'].astype(str).str.strip() != '')
df_raw['is_multibank']        = df_raw['has_other_simpan'] | df_raw['has_other_transaksi']

n_multibank = int(df_raw['is_multibank'].sum())
n_single    = total - n_multibank

multi_recs = [
    {'kategori': 'Single Bank',  'bank_tambahan': '-',             'n': n_single,    'pct': round(n_single/total*100, 1),    'tipe': 'Eksklusif XYZ'},
    {'kategori': 'Multi Bank',   'bank_tambahan': 'Ada bank lain', 'n': n_multibank, 'pct': round(n_multibank/total*100, 1), 'tipe': 'Multibank'},
]

for bank, cnt in df_raw[df_raw['has_other_simpan']]['A1XX'].value_counts().head(6).items():
    if str(bank).strip():
        multi_recs.append({'kategori': 'Bank Simpan Tambahan',    'bank_tambahan': str(bank).strip(), 'n': int(cnt), 'pct': round(cnt/total*100, 1), 'tipe': 'Simpan'})

for bank, cnt in df_raw[df_raw['has_other_transaksi']]['A1X'].value_counts().head(6).items():
    if str(bank).strip():
        multi_recs.append({'kategori': 'Bank Transaksi Tambahan', 'bank_tambahan': str(bank).strip(), 'n': int(cnt), 'pct': round(cnt/total*100, 1), 'tipe': 'Transaksi'})

multi_df = pd.DataFrame(multi_recs)
multi_df.to_csv(os.path.join(DATA_DIR, 'multibank_analysis.csv'), index=False)
print('multibank_analysis.csv saved:')
print(multi_df)

multibank_analysis.csv saved:
                   kategori                bank_tambahan     n   pct  \
0               Single Bank                            -  1184  68.4   
1                Multi Bank                Ada bank lain   546  31.6   
2      Bank Simpan Tambahan      Bank Central Asia (BCA)   188  10.9   
3      Bank Simpan Tambahan  Bank Rakyat Indonesia (BRI)   148   8.6   
4      Bank Simpan Tambahan                 Bank Mandiri    84   4.9   
5      Bank Simpan Tambahan  Bank Negara Indonesia (BNI)    59   3.4   
6      Bank Simpan Tambahan                      Lainnya    43   2.5   
7      Bank Simpan Tambahan   Bank Tabungan Negara (BTN)     8   0.5   
8   Bank Transaksi Tambahan      Bank Central Asia (BCA)   188  10.9   
9   Bank Transaksi Tambahan  Bank Rakyat Indonesia (BRI)   148   8.6   
10  Bank Transaksi Tambahan                 Bank Mandiri    84   4.9   
11  Bank Transaksi Tambahan  Bank Negara Indonesia (BNI)    59   3.4   
12  Bank Transaksi Tambahan       

In [14]:
import pandas as pd, os

DATA_DIR = r'C:\Users\M S I\OneDrive\Desktop\project-bankxyz\data'

for f in os.listdir(DATA_DIR):
    if f.endswith('.csv'):
        df = pd.read_csv(os.path.join(DATA_DIR, f), nrows=1)
        print(f'{f}: {df.columns.tolist()}')

agg_branch.csv: ['PROV', 'KABKOTA', 'CABANG', 'nps_score', 'csi_num', 'loyalty_num', 'emotion_positive_score', 'emotion_negative_score', 'emotion_net', 'n_responden', 'ovr_operasional', 'ovr_parkir', 'ovr_banking_hall', 'ovr_toilet', 'ovr_sekuriti', 'ovr_teller', 'ovr_cs']
agg_gender.csv: ['gender', 'nps_score', 'csi_mean', 'loyalty_mean', 'n']
agg_panel.csv: ['panel', 'nps_score', 'csi_mean', 'loyalty_mean', 'n']
agg_provinsi.csv: ['PROV', 'nps_score', 'csi_num', 'loyalty_num', 'emotion_positive_score', 'emotion_negative_score', 'emotion_net', 'n_responden', 'ovr_operasional', 'ovr_parkir', 'ovr_banking_hall', 'ovr_toilet', 'ovr_sekuriti', 'ovr_teller', 'ovr_cs']
agg_segmen.csv: ['segmen', 'nps_score', 'csi_mean', 'n']
agg_usia.csv: ['usia_group', 'nps_score', 'csi_mean', 'loyalty_mean', 'n']
brand_perception.csv: ['atribut', 'xyz_pct_agree', 'komp_pct_agree', 'selisih']
competitor_benchmark.csv: ['atribut', 'mean_score', 'pct_agree', 'n']
comp_nps_per_provinsi.csv: ['provinsi', 'xyz_